In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Vstupy a výstupy primitiv

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';





<Accordion>
<AccordionItem title="Verze balíčků">

Kód na této stránce byl vyvinut s použitím následujících požadavků.
Doporučujeme používat tyto verze nebo novější.

```
qiskit[all]~=2.4.0
```
</AccordionItem>
</Accordion>
Tato stránka poskytuje přehled vstupů a výstupů primitiv Qiskit. S těmito primitivy můžeš použít datovou strukturu zvanou **Primitive Unified Bloc (PUB)** k efektivnímu definování vektorizovaných úloh. Tyto PUBs jsou základní jednotkou práce pro spouštění úloh. Používají se jako vstupy do metody `run()` primitiv Sampler a Estimator, které spustí definovanou úlohu jako job. Po dokončení jobu jsou výsledky vráceny ve formátu, který závisí na použitých PUBs a případně zadaných možnostech.
<span id="pubs"></span>
## Přehled PUBs
Při volání metody `run()` primitiva je hlavním požadovaným argumentem `list` jedné nebo více n-tic – jedna pro každý obvod spouštěný primitivem. Každá z těchto n-tic je považována za PUB a požadované prvky každé n-tice v seznamu závisí na použitém primitivu. Data poskytnutá těmto n-ticím mohou být také uspořádána v různých tvarech, což umožňuje flexibilitu v úloze prostřednictvím broadcastingu – jehož pravidla jsou popsána v [následující části](#broadcasting-rules).

### Estimator PUB
Pro primitiv Estimator by formát PUB měl obsahovat nejvýše čtyři hodnoty:
- Jediný `QuantumCircuit`, který může obsahovat jeden nebo více objektů [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
- Seznam jednoho nebo více pozorovatelných veličin, které určují střední hodnoty k odhadnutí, uspořádaných do pole (například jedna pozorovatelná veličina reprezentovaná jako 0-d pole, seznam pozorovatelných veličin jako 1-d pole atd.). Data mohou být v libovolném formátu `ObservablesArrayLike`, například `Pauli`, `SparsePauliOp`, `PauliList` nebo `str`.
   > **Note:** Pokud máš dvě komutující pozorovatelné veličiny v různých PUBs, ale se stejným Circuit, nebudou odhadovány pomocí stejného měření. Každý PUB představuje jiný základ pro měření, a proto jsou pro každý PUB nutná samostatná měření. Aby byly komutující pozorovatelné veličiny odhadovány pomocí stejného měření, musí být seskupeny v rámci stejného PUB.
- Kolekce hodnot parametrů, na které se Circuit naváže. Tuto kolekci lze zadat jako jediný objekt podobný poli, kde poslední index odpovídá objektům `Parameter` Circuit, nebo ji lze vynechat (případně nastavit na `None`), pokud Circuit neobsahuje žádné objekty `Parameter`.
- (Volitelně) cílová přesnost pro odhadování středních hodnot

### Sampler PUB
Pro primitiv Sampler obsahuje formát n-tice PUB nejvýše tři hodnoty:
- Jediný `QuantumCircuit`, který může obsahovat jeden nebo více objektů [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
   *Poznámka: Tyto Circuits musí také obsahovat instrukce pro měření každého z Qubitů, které mají být vzorkovány.*
- Kolekce hodnot parametrů, na které se Circuit naváže $\theta_k$ (potřebná pouze tehdy, pokud jsou použity nějaké objekty `Parameter`, které musí být navázány za běhu)
- (Volitelně) počet shotů, se kterými se Circuit změří
---

Následující kód demonstruje příkladnou sadu vektorizovaných vstupů pro primitiv `Estimator`.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
    ClassicalRegister,
    QuantumRegister,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.containers import BitArray
from qiskit.primitives import StatevectorEstimator


import numpy as np

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit without providing a backend
pm = generate_preset_pass_manager(optimization_level=1)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 10),
        np.linspace(-4 * np.pi, 4 * np.pi, 10),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator = StatevectorEstimator()
estimator_pub = (transpiled_circuit, observables, params)

# Run the transpiled circuit
# using the set of parameters and observables.

job = estimator.run([estimator_pub])
result = job.result()

<span id="broadcasting"></span>
### Broadcasting rules

The PUBs aggregate elements from multiple arrays (observables and parameter values) by following the same broadcasting rules as NumPy. This section briefly summarizes those rules.  For a detailed explanation, see the [NumPy broadcasting rules documentation](https://numpy.org/doc/stable/user/basics.broadcasting.html).

Rules:

* Input arrays do not need to have the same number of dimensions.
  * The resulting array will have the same number of dimensions as the input array with the largest dimension.
  * The size of each dimension is the largest size of the corresponding dimension.
  * Missing dimensions are assumed to have size one.
* Shape comparisons start with the rightmost dimension and continue to the left.
* Two dimensions are compatible if their sizes are equal or if one of them is 1.

Examples of array pairs that broadcast:

```text
A1     (1d array):      1
A2     (2d array):  3 x 5
Result (2d array):  3 x 5


A1     (3d array):  11 x 2 x 7
A2     (3d array):  11 x 1 x 7
Result (3d array):  11 x 2 x 7
```

Examples of array pairs that do not broadcast:

```text
A1     (1d array):  5
A2     (1d array):  3

A1     (2d array):      2 x 1
A2     (3d array):  6 x 5 x 4 # This would work if the middle dimension were 2, but it is 5.
```

`Estimator` returns one expectation value estimate for each element of the broadcasted shape.

Here are some examples of common patterns expressed in terms of array broadcasting.  Their accompanying visual representation is shown in the figure that follows:


Parameter value sets are represented by n x m arrays, and observable arrays are represented by one or more single-column arrays. For each example in the previous code, the parameter value sets are combined with their observable array to create the resulting expectation value estimates.

 - *Example 1*: (broadcast single observable) has a parameter value set that is a 5x1 array and a 1x1 observables array.  The one item in the observables array is combined with each item in the parameter value set to create a single 5x1 array where each item is a combination of the original item in the parameter value set with the item in the observables array.

 - *Example 2*: (zip) has a 5x1 parameter value set and a 5x1 observables array.  The output is a 5x1 array where each item is a combination of the nth item in the parameter value set with the nth item in the observables array.

  - *Example 3*: (outer/product) has a 1x6 parameter value set and a 4x1 observables array.  Their combination results in a 4x6 array that is created by combining each item in the parameter value set with *every* item in the observables array, and thus each parameter value becomes an entire column in the output.

  - *Example 4*: (Standard nd generalization) has a 3x6 parameter value set array and two 3x1 observables array.  These combine to create two 3x6 output arrays in a similar manner to the previous example.

![This image illustrates several visual representations of array broadcasting.](../docs/images/guides/primitive-input-output/broadcasting.svg "Visual representation of broadcasting")

In [2]:
# Broadcast single observable
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = SparsePauliOp("ZZZ")  # shape ()
# >> pub result has shape (5,)

# Zip
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = [
    SparsePauliOp(pauli) for pauli in ["III", "XXX", "YYY", "ZZZ", "XYZ"]
]  # shape (5,)
# >> pub result has shape (5,)

# Outer/Product
parameter_values = np.random.uniform(size=(1, 6))  # shape (1, 6)
observables = [
    [SparsePauliOp(pauli)] for pauli in ["III", "XXX", "YYY", "ZZZ"]
]  # shape (4, 1)
# >> pub result has shape (4, 6)

# Standard nd generalization
parameter_values = np.random.uniform(size=(3, 6))  # shape (3, 6)
observables = [
    [
        [SparsePauliOp(["XII"])],
        [SparsePauliOp(["IXI"])],
        [SparsePauliOp(["IIX"])],
    ],
    [
        [SparsePauliOp(["ZII"])],
        [SparsePauliOp(["IZI"])],
        [SparsePauliOp(["IIZ"])],
    ],
]  # shape (2, 3, 1)
# >> pub result has shape (2, 3, 6)

<Admonition type="tip" title="SparsePauliOp">
Each `SparsePauliOp` counts as a single element in this context, regardless of the number of Paulis contained in the `SparsePauliOp`. Thus, for the purpose of these broadcasting rules, all of the following elements have the same shape:

```text
a = SparsePauliOp("Z") # shape ()
b = SparsePauliOp("IIIIZXYIZ") # shape ()
c = SparsePauliOp.from_list(["XX", "XY", "IZ"]) # shape ()
```

The following lists of operators, while equivalent in terms of information contained, have different shapes:

```text
list1 = SparsePauliOp.from_list(["XX", "XY", "IZ"]) # shape ()
list2 = [SparsePauliOp("XX"), SparsePauliOp("XY"), SparsePauliOp("IZ")] # shape (3, )
```
</Admonition>

## Overview of primitive outputs

Once one or more PUBs are sent to a QPU for execution and a job successfully completes, the data is returned as a [`PrimitiveResult`](/docs/api/qiskit/qiskit.primitives.PrimitiveResult) container object. The `PrimitiveResult` contains an iterable list of [`PubResult`](/docs/api/qiskit/qiskit.primitives.PubResult) objects that contain the execution results for each PUB. For example, a job submitted with 20 PUBs will return a `PrimitiveResult` object that contains a list of 20 `PubResults`, one corresponding to each PUB.

Each of these `PubResult` objects possess both a `data` and an optional `metadata` attribute. The `data` attribute is a customized [`DataBin`](/docs/api/qiskit/qiskit.primitives.DataBin) that contains the the expectation value estimates in case of the Estimator, or samples of the circuit output in the case of the Sampler.

The `data` attribute might also include other implementation-specific information such as standard deviations. The `metadata` attribute can contain additional implementation-specific information about the execution of the associated PUB.

The following is a visual outline of the `PrimitiveResult` data structure:

<Tabs>
    <TabItem value="estimator" label="Estimator output">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object,
        |       |     ## which includes data such as the following:
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object,
        |       |     ## which includes data such as the following:
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```
    <Admonition type="note">
    The above is an example of data that might be returned.  The actual data returned depends on the implementation.
    </Admonition>
    </TabItem>
    <TabItem value="sampler" label="Sampler output">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── NAME_OF_CLASSICAL_REGISTER
        │       │   └── BitArray of count data for first PUB (default is 'meas')
        |       |
        │       └── NAME_OF_ANOTHER_CLASSICAL_REGISTER
        │           └── BitArray of count data (exists only if more than one
        |                 ClassicalRegister was specified in the circuit)
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       └── NAME_OF_CLASSICAL_REGISTER
        |           └── BitArray of count data for second PUB
        ├── ...
        ├── ...
        └── ...
    ```
    </TabItem>
</Tabs>

### Estimator output

As stated previously, the data returned in the `PubResult` for the Estimator primitive depends on the implementation. For example, it might contain an array of expectation values (`PubResult.data.evs`) and associated standard deviations (`PubResult.data.stds`).

The below code snippet describes the `PrimitiveResult` (and associated `PubResult`) format for the job created above.

In [3]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 10), dtype=float64>), stds=np.ndarray(<shape=(3, 10), dtype=float64>), shape=(3, 10)), metadata={'target_precision': 0.0, 'circuit_metadata': {}})], metadata={'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 10), dtype=float64>), stds=np.ndarray(<shape=(3, 10), dtype=float64>), shape=(3, 10))

And this DataBin has attributes: dict_keys(['evs', 'stds'])
Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). 

The expectation values measured from this PUB are: 
[[ 3.06161700e-16  4.52395120e-01  4.36594428e-01  2.16506351e-01
   6.33718361e-01 -6.33718361e-01 -2.16506351e-01 -4.36594428e-01
  -4.52395120e-01 -3.06161700e-16]
 [ 1.22464680

### Sampler output

When a Sampler job is completed successfully, the returned [`PrimitiveResult`](/docs/api/qiskit/qiskit.primitives.PrimitiveResult) object contains a list of [`SamplerPubResult`](/docs/api/qiskit/qiskit.primitives.SamplerPubResult)s, one per PUB. The data bins of these `SamplerPubResult` objects are dict-like objects that contain one `BitArray` per `ClassicalRegister` in the circuit.

The `BitArray` class is a container for ordered shot data. In more detail, it stores the sampled bitstrings as bytes inside a two-dimensional array. The left-most axis of this array runs over ordered shots, while the right-most axis runs over bytes.

As a first example, let us look at the following ten-qubit circuit:

In [4]:
from qiskit.primitives import StatevectorSampler

# generate a ten-qubit GHZ circuit
circuit = QuantumCircuit(10)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure_all()

# transpile the circuit
transpiled_circuit = pm.run(circuit)

sampler = StatevectorSampler()

# run the Sampler job and retrieve the results

job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains one BitArray
data = result[0].data
print(f"Databin: {data}\n")

# to access the BitArray, use the key "meas", which is the default name of
# the classical register when this is added by the `measure_all` method
array = data.meas
print(f"BitArray: {array}\n")
print(f"The shape of register `meas` is {data.meas.array.shape}.\n")
print(f"The bytes in register `alpha`, shot by shot:\n{data.meas.array}\n")

Databin: DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=10>))

BitArray: BitArray(<shape=(), num_shots=1024, num_bits=10>)

The shape of register `meas` is (1024, 2).

The bytes in register `alpha`, shot by shot:
[[  0   0]
 [  3 255]
 [  0   0]
 ...
 [  3 255]
 [  3 255]
 [  3 255]]



<span id="broadcasting"></span>
### Pravidla broadcastingu
PUBy agregují prvky z více polí (observables a hodnoty parametrů) podle stejných pravidel broadcastingu jako NumPy. Tato sekce tato pravidla stručně shrnuje. Podrobné vysvětlení najdeš v [dokumentaci NumPy k pravidlům broadcastingu](https://numpy.org/doc/stable/user/basics.broadcasting.html).

Pravidla:

* Vstupní pole nemusí mít stejný počet dimenzí.
  * Výsledné pole bude mít stejný počet dimenzí jako vstupní pole s největším počtem dimenzí.
  * Velikost každé dimenze je největší velikost odpovídající dimenze.
  * Chybějící dimenze se považují za dimenze o velikosti jedna.
* Porovnávání tvarů začíná od nejpravější dimenze a pokračuje doleva.
* Dvě dimenze jsou kompatibilní, pokud jsou jejich velikosti stejné nebo pokud je jedna z nich 1.

`Estimator` vrátí jeden odhad střední hodnoty pro každý prvek broadcastovaného tvaru.

Níže jsou uvedeny příklady běžných vzorů vyjádřených pomocí broadcastingu polí. Jejich vizuální reprezentace je zobrazena na obrázku, který za nimi následuje:

Sady hodnot parametrů jsou reprezentovány poli n x m a pole observables jsou reprezentovány jedním nebo více jednosloupcovými poli. Pro každý příklad v předchozím kódu jsou sady hodnot parametrů kombinovány s příslušným polem observables za účelem vytvoření výsledných odhadů střední hodnoty.

 - *Příklad 1*: (broadcast jednoho observable) má sadu hodnot parametrů tvaru 5x1 a pole observables tvaru 1x1. Jeden prvek v poli observables je kombinován s každým prvkem sady hodnot parametrů, čímž vznikne jedno pole 5x1, kde každý prvek je kombinací původního prvku sady hodnot parametrů s prvkem pole observables.

 - *Příklad 2*: (zip) má sadu hodnot parametrů 5x1 a pole observables 5x1. Výstupem je pole 5x1, kde každý prvek je kombinací n-tého prvku sady hodnot parametrů s n-tým prvkem pole observables.

  - *Příklad 3*: (outer/product) má sadu hodnot parametrů 1x6 a pole observables 4x1. Jejich kombinací vznikne pole 4x6, vytvořené zkombinováním každého prvku sady hodnot parametrů s *každým* prvkem pole observables – každá hodnota parametru se tak stane celým sloupcem ve výstupu.

  - *Příklad 4*: (standardní nd zobecnění) má pole sady hodnot parametrů 3x6 a dvě pole observables 3x1. Tato pole se kombinují a vznikají dvě výstupní pole 3x6 podobným způsobem jako v předchozím příkladu.

![Tento obrázek znázorňuje několik vizuálních reprezentací broadcastingu polí.](../docs/images/guides/primitive-input-output/broadcasting.svg "Vizuální reprezentace broadcastingu")

:::tip[SparsePauliOp]

Každý `SparsePauliOp` se v tomto kontextu počítá jako jeden prvek, bez ohledu na počet Pauliho operátorů, které `SparsePauliOp` obsahuje. Pro účely těchto pravidel broadcastingu mají tedy všechny následující prvky stejný tvar:

In [5]:
# optionally, convert away from the native BitArray format to a dictionary format
counts = data.meas.get_counts()
print(f"Counts: {counts}")

Counts: {'0000000000': 492, '1111111111': 532}


When a circuit contains more than one classical register, the results are stored in different `BitArray` objects. The following example modifies the previous snippet by splitting the classical register into two distinct registers:

In [6]:
# generate a ten-qubit GHZ circuit with two classical registers
circuit = QuantumCircuit(
    qreg := QuantumRegister(10),
    alpha := ClassicalRegister(1, "alpha"),
    beta := ClassicalRegister(9, "beta"),
)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure([0], alpha)
circuit.measure(range(1, 10), beta)

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results

job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains two BitArrays, one per register, and can be accessed
# as attributes using the registers' names
data = result[0].data
print(f"BitArray for register 'alpha': {data.alpha}")
print(f"BitArray for register 'beta': {data.beta}")

BitArray for register 'alpha': BitArray(<shape=(), num_shots=1024, num_bits=1>)
BitArray for register 'beta': BitArray(<shape=(), num_shots=1024, num_bits=9>)


:::

## Přehled výstupů primitiv
Jakmile jsou jeden nebo více PUBů odeslány na QPU k vykonání a úloha se úspěšně dokončí, data se vrátí jako kontejnerový objekt [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult). `PrimitiveResult` obsahuje iterovatelný seznam objektů [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult), které obsahují výsledky vykonání pro každý PUB. Například úloha odeslaná s 20 PUBy vrátí objekt `PrimitiveResult` obsahující seznam 20 `PubResult`ů, přičemž každý odpovídá jednomu PUBu.

Každý z těchto objektů `PubResult` má atribut `data` a volitelný atribut `metadata`. Atribut `data` je přizpůsobený [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin), který obsahuje odhady očekávaných hodnot v případě Estimatoru, nebo vzorky výstupu Circuit v případě Sampleru.

Atribut `data` může také obsahovat další informace specifické pro danou implementaci, jako jsou směrodatné odchylky. Atribut `metadata` může obsahovat další informace specifické pro implementaci o spuštění příslušného PUBu.

Níže je znázorněna vizuální struktura dat `PrimitiveResult`:

<Tabs>
    <TabItem value="estimator" label="Výstup Estimatoru">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object,
        |       |     ## which includes data such as the following:
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object,
        |       |     ## which includes data such as the following:
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```
    > **Note:** Výše uvedený příklad znázorňuje data, která mohou být vrácena. Skutečně vrácená data závisí na implementaci.
    </TabItem>
    <TabItem value="sampler" label="Výstup Sampleru">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── NAME_OF_CLASSICAL_REGISTER
        │       │   └── BitArray of count data for first PUB (default is 'meas')
        |       |
        │       └── NAME_OF_ANOTHER_CLASSICAL_REGISTER
        │           └── BitArray of count data (exists only if more than one
        |                 ClassicalRegister was specified in the circuit)
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       └── NAME_OF_CLASSICAL_REGISTER
        |           └── BitArray of count data for second PUB
        ├── ...
        ├── ...
        └── ...
    ```
    </TabItem>
</Tabs>

### Výstup Estimatoru
Jak bylo uvedeno výše, data vrácená v `PubResult` pro primitiv Estimator závisí na implementaci. Může například obsahovat pole očekávaných hodnot (`PubResult.data.evs`) a příslušné směrodatné odchylky (`PubResult.data.stds`).

Níže uvedený úryvek kódu popisuje formát `PrimitiveResult` (a příslušného `PubResult`) pro výše vytvořenou úlohu.

In [7]:
print(f"The shape of register `alpha` is {data.alpha.array.shape}.")
print(f"The bytes in register `alpha`, shot by shot:\n{data.alpha.array}\n")

print(f"The shape of register `beta` is {data.beta.array.shape}.")
print(f"The bytes in register `beta`, shot by shot:\n{data.beta.array}\n")

# post-select the bitstrings of `beta` based on having sampled "1" in `alpha`
mask = data.alpha.array == "0b1"
ps_beta = data.beta[mask[:, 0]]
print(f"The shape of `beta` after post-selection is {ps_beta.array.shape}.")
print(f"The bytes in `beta` after post-selection:\n{ps_beta.array}")

# get a slice of `beta` to retrieve the first three bits
beta_sl_bits = data.beta.slice_bits([0, 1, 2])
print(
    f"The shape of `beta` after bit-wise slicing is {beta_sl_bits.array.shape}."
)
print(f"The bytes in `beta` after bit-wise slicing:\n{beta_sl_bits.array}\n")

# get a slice of `beta` to retrieve the bytes of the first five shots
beta_sl_shots = data.beta.slice_shots([0, 1, 2, 3, 4])
print(
    f"The shape of `beta` after shot-wise slicing is {beta_sl_shots.array.shape}."
)
print(
    f"The bytes in `beta` after shot-wise slicing:\n{beta_sl_shots.array}\n"
)

# calculate the expectation value of diagonal operators on `beta`
ops = [SparsePauliOp("ZZZZZZZZZ"), SparsePauliOp("IIIIIIIIZ")]
exp_vals = data.beta.expectation_values(ops)
for o, e in zip(ops, exp_vals):
    print(f"Exp. val. for observable `{o}` is: {e}")

# concatenate the bitstrings in `alpha` and `beta` to "merge" the results of the two
# registers
merged_results = BitArray.concatenate_bits([data.alpha, data.beta])
print(f"\nThe shape of the merged results is {merged_results.array.shape}.")
print(f"The bytes of the merged results:\n{merged_results.array}\n")

The shape of register `alpha` is (1024, 1).
The bytes in register `alpha`, shot by shot:
[[1]
 [1]
 [1]
 ...
 [0]
 [0]
 [1]]

The shape of register `beta` is (1024, 2).
The bytes in register `beta`, shot by shot:
[[  1 255]
 [  1 255]
 [  1 255]
 ...
 [  0   0]
 [  0   0]
 [  1 255]]

The shape of `beta` after post-selection is (0, 2).
The bytes in `beta` after post-selection:
[]
The shape of `beta` after bit-wise slicing is (1024, 1).
The bytes in `beta` after bit-wise slicing:
[[7]
 [7]
 [7]
 ...
 [0]
 [0]
 [7]]

The shape of `beta` after shot-wise slicing is (5, 2).
The bytes in `beta` after shot-wise slicing:
[[  1 255]
 [  1 255]
 [  1 255]
 [  1 255]
 [  1 255]]



Exp. val. for observable `SparsePauliOp(['ZZZZZZZZZ'],
              coeffs=[1.+0.j])` is: -0.017578125
Exp. val. for observable `SparsePauliOp(['IIIIIIIIZ'],
              coeffs=[1.+0.j])` is: -0.017578125

The shape of the merged results is (1024, 2).
The bytes of the merged results:
[[  3 255]
 [  3 255]
 [  3 255]
 ...
 [  0   0]
 [  0   0]
 [  3 255]]



## Result metadata

In addition to the execution results, the `PrimitiveResult` and `PubResult` objects contain an optional metadata attribute about the job that was submitted. The metadata returned (if any) is implementation-specific.

In [8]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'version' : 2,

The metadata of the PubResult result is:
'shots' : 1024,
'circuit_metadata' : {},


## Next steps

<Admonition type="tip" title="Recommendations">

  -  Review the [Qiskit primitives](/docs/api/qiskit/primitives) API.
  -  Review the [Qiskit Aer primitives](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html) API.
  -  Learn more about the [Qiskit Runtime primitives](/docs/guides/qiskit-runtime-primitives).
  -  Review the [Qiskit Runtime Estimator](/docs/api/qiskit-ibm-runtime/estimator-v2) API.
  -  Review the [Qiskit Runtime Sampler](/docs/api/qiskit-ibm-runtime/sampler-v2) API.
</Admonition>